In [1]:
import duckdb

con = duckdb.connect()

Criação de Views

In [9]:
con.execute("CREATE VIEW orders AS SELECT * FROM '../data/raw/olist_orders_dataset.csv'")
con.execute("CREATE VIEW customers AS SELECT * FROM '../data/raw/olist_customers_dataset.csv'")
con.execute("CREATE VIEW order_items AS SELECT * FROM '../data/raw/olist_order_items_dataset.csv'")
con.execute("CREATE VIEW payments AS SELECT * FROM '../data/raw/olist_order_payments_dataset.csv'")
con.execute("CREATE VIEW reviews AS SELECT * FROM '../data/raw/olist_order_reviews_dataset.csv'")
con.execute("CREATE VIEW products AS SELECT * FROM '../data/raw/olist_products_dataset.csv'")
con.execute("CREATE VIEW sellers AS SELECT * FROM '../data/raw/olist_sellers_dataset.csv'")
con.execute("CREATE VIEW geolocation AS SELECT * FROM '../data/raw/olist_geolocation_dataset.csv'")
con.execute("CREATE VIEW category_translation AS SELECT * FROM '../data/raw/product_category_name_translation.csv'")

In [10]:
con.sql("SELECT * FROM orders LIMIT 5").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


Então nesse conjunto 'orders' temos os pedidos com algumas colunas que vão ser importantes para a gente: quando foi entregue e a data de entrega que esperada 

In [27]:
con.sql("SELECT COUNT(*) as total_pedidos FROM orders").df()

,total_pedidos
0,99441


In [15]:
con.execute("SELECT DISTINCT(order_status) FROM orders").df()

,order_status
0,delivered
1,created
2,canceled
3,invoiced
4,unavailable
5,approved
6,processing
7,shipped


Quando for trabalhar com os dados temos que selecionar somente as 'delivered'

In [16]:
con.execute("SELECT order_status, COUNT(*) as total_pedidos FROM orders GROUP BY order_status").df()

,order_status,total_pedidos
0,processing,301
1,shipped,1107
2,delivered,96478
3,created,5
4,unavailable,609
5,approved,2
6,invoiced,314
7,canceled,625


Proporção de nulos das colunas:

In [21]:
con.execute("SELECT COUNT(order_delivered_customer_date)/COUNT(*) * 100 as data_entrega_nao_nula FROM orders WHERE order_status = 'delivered'").df()

,data_entrega_nao_nula
0,99.991708


In [22]:
con.execute("SELECT COUNT(order_estimated_delivery_date)/COUNT(*) * 100 as data_entrega_prevista_nao_nula FROM orders WHERE order_status = 'delivered'").df()

,data_entrega_prevista_nao_nula
0,100.0


Temos poucas informações nulas isso é ótimo (vamos removelas)

Vamos verificar a cardinalidade das chaves agora:

In [25]:
con.execute("SELECT COUNT(DISTINCT order_id)/COUNT(*) as prop_pedidos_distintos, COUNT(DISTINCT customer_id)/COUNT(*) as prop_clientes_distintos FROM orders").df()

,prop_pedidos_distintos,prop_clientes_distintos
0,1.0,1.0


Ótimo que pedidos tenham chaves diferentes, mas estranho não ter um cliente que se repete

Dei uma olhada aqui e o customer_id é gerado por pedido, não por pessoa. Veremos mais na frente que a tabela customers tem uma coluna separada, customer_unique_id, que é o identificador real da pessoa

Pra poder classificar em atraso e cumprimento vamos precisar ver onde a data de entrega foi maior do que a data prevista

In [26]:
con.execute("SELECT CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Atrasou' ELSE 'Cumpriu' END as status_entrega, COUNT(*) as total_pedidos FROM orders WHERE order_status = 'delivered' GROUP BY status_entrega").df()

,status_entrega,total_pedidos
0,Cumpriu,88652
1,Atrasou,7826


Fechou, tenho quase 8000 atrasos. Já vi que tem o desbalanceamento de classes como esperado.

Agora eu vou precisar avaliar as outras bases de dados para saber quais variáveis eu tenho interesse em utilizar no meu modelo.

In [30]:
import pandas as pd

def auditar_tabela(con, nome_tabela):
    print(f"=== {nome_tabela} ===")
    
    schema = con.sql(f"DESCRIBE {nome_tabela}").df()
    colunas = schema['column_name'].tolist()
    
    total_linhas = con.sql(f"SELECT COUNT(*) as n FROM {nome_tabela}").df()['n'][0]
    duplicatas = total_linhas - con.sql(f"SELECT COUNT(*) as n FROM (SELECT DISTINCT * FROM {nome_tabela})").df()['n'][0]
    print(f"Linhas: {total_linhas} | Linhas duplicadas (100% iguais): {duplicatas}\n")
    
    selects = [
        f"ROUND(100.0 * SUM(CASE WHEN \"{col}\" IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS \"{col}__pct_nulos\", "
        f"ROUND(100.0 * COUNT(DISTINCT \"{col}\")/ COUNT(*), 2) AS \"{col}__distintos\""
        for col in colunas
    ]
    resultado = con.sql(f"SELECT {', '.join(selects)} FROM {nome_tabela}").df()
    
    resumo = pd.DataFrame({
        "coluna": colunas,
        "tipo": schema['column_type'].tolist(),
        "pct_nulos": [resultado[f"{c}__pct_nulos"][0] for c in colunas],
        "valores_distintos": [resultado[f"{c}__distintos"][0] for c in colunas],
    })
    
    display(resumo)
    return resumo

In [31]:
tabelas = ["orders", "customers", "order_items", "payments", "reviews", 
           "products", "sellers", "geolocation", "category_translation"]

resumos = {}
for tabela in tabelas:
    resumos[tabela] = auditar_tabela(con, tabela)

=== orders ===
Linhas: 99441 | Linhas duplicadas (100% iguais): 0



,coluna,tipo,pct_nulos,valores_distintos
0,order_id,VARCHAR,0.00,100.00
1,customer_id,VARCHAR,0.00,100.00
2,order_status,VARCHAR,0.00,0.01
3,order_purchase_timestamp,TIMESTAMP,0.00,99.43
4,order_approved_at,TIMESTAMP,0.16,91.24
5,order_delivered_carrier_date,TIMESTAMP,1.79,81.47
6,order_delivered_customer_date,TIMESTAMP,2.98,96.20
7,order_estimated_delivery_date,TIMESTAMP,0.00,0.46


=== customers ===
Linhas: 99441 | Linhas duplicadas (100% iguais): 0



,coluna,tipo,pct_nulos,valores_distintos
0,customer_id,VARCHAR,0.0,100.00
1,customer_unique_id,VARCHAR,0.0,96.64
2,customer_zip_code_prefix,VARCHAR,0.0,15.08
3,customer_city,VARCHAR,0.0,4.14
4,customer_state,VARCHAR,0.0,0.03


=== order_items ===
Linhas: 112650 | Linhas duplicadas (100% iguais): 0



...

,coluna,tipo,pct_nulos,valores_distintos
0,seller_id,VARCHAR,0.0,100.00
1,seller_zip_code_prefix,VARCHAR,0.0,72.57
2,seller_city,VARCHAR,0.0,19.74
3,seller_state,VARCHAR,0.0,0.74


=== geolocation ===
Linhas: 1000163 | Linhas duplicadas (100% iguais): 261831



,coluna,tipo,pct_nulos,valores_distintos
0,geolocation_zip_code_prefix,VARCHAR,0.0,1.90
1,geolocation_lat,DOUBLE,0.0,71.73
2,geolocation_lng,DOUBLE,0.0,71.75
3,geolocation_city,VARCHAR,0.0,0.80
4,geolocation_state,VARCHAR,0.0,0.00


=== category_translation ===
Linhas: 71 | Linhas duplicadas (100% iguais): 0



,coluna,tipo,pct_nulos,valores_distintos
0,product_category_name,VARCHAR,0.0,100.0
1,product_category_name_english,VARCHAR,0.0,100.0
